# Notebook 03 — LangGraph 에이전트 구성

## 목표
- LangGraph StateGraph로 DMS 파이프라인 구성
- 5개 에이전트 노드 구현
- Ollama 로컬 LLM 연동
- 에이전트 간 상태(State) 흐름 이해

## 스킬업 포인트
- TypedDict로 LangGraph State 설계
- Node = 순수 함수 (state in → state out)
- Edge = 조건부 라우팅
- Ollama = 무료 로컬 LLM (API 비용 0원)

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import sys, time, subprocess
from typing import TypedDict, Optional, List, Annotated
from dataclasses import dataclass, field

import cv2
import numpy as np
from scipy.spatial import distance as dist
import mediapipe as mp
from collections import deque

from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

print('패키지 로드 완료')

## 1. LangGraph State 설계

LangGraph에서 State는 **에이전트 간에 공유되는 유일한 데이터 통로**입니다.

```
Node A → State 업데이트 → Node B → State 업데이트 → ...
```

TypedDict로 정의해야 LangGraph가 타입 추론할 수 있어요.

In [ ]:
class DMSState(TypedDict):
    """
    DMS 에이전트 파이프라인 공유 상태
    모든 에이전트가 이 State를 읽고 업데이트합니다.
    """
    # 입력
    frame:          Optional[np.ndarray]   # 현재 프레임
    frame_id:       int                    # 프레임 번호

    # Face Analysis 결과
    face_detected:  bool
    ear:            Optional[float]
    mar:            Optional[float]
    pitch:          Optional[float]
    yaw:            Optional[float]
    perclos:        Optional[float]

    # Object Detection 결과
    detected_objects: List[dict]

    # State Classifier 결과
    is_drowsy:      bool
    is_yawning:     bool
    is_distracted:  bool
    has_danger_obj: bool
    risk_count:     int

    # Alert Manager 결과
    alert_level:    int     # 0=정상, 1=주의, 2=경고, 3=위험
    alert_reason:   str

    # LLM Reasoning 결과
    llm_message:    str

    # 히스토리 (PERCLOS용)
    ear_history:    List[float]


def initial_state(frame=None, frame_id=0) -> DMSState:
    """초기 State 생성"""
    return DMSState(
        frame=frame, frame_id=frame_id,
        face_detected=False,
        ear=None, mar=None, pitch=None, yaw=None, perclos=None,
        detected_objects=[],
        is_drowsy=False, is_yawning=False,
        is_distracted=False, has_danger_obj=False,
        risk_count=0,
        alert_level=0, alert_reason='정상',
        llm_message='',
        ear_history=[]
    )

print('DMSState 정의 완료')
print('State 키 목록:', list(DMSState.__annotations__.keys()))

## 2. 에이전트 노드 구현

각 노드는 `state: DMSState → DMSState` 형태의 순수 함수입니다.

### 노드 1: Face Analysis Agent

In [ ]:
# 공통 상수
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH     = [61, 291, 13, 14, 17, 0, 402, 178]
EAR_THRESH = 0.25
MAR_THRESH = 0.60

def _ear(pts):
    A=dist.euclidean(pts[1],pts[5]); B=dist.euclidean(pts[2],pts[4])
    C=dist.euclidean(pts[0],pts[3])
    return (A+B)/(2*C)

def _mar(pts):
    A=dist.euclidean(pts[1],pts[7]); B=dist.euclidean(pts[2],pts[6])
    C=dist.euclidean(pts[3],pts[5]); D=dist.euclidean(pts[0],pts[4])
    return (A+B+C)/(2*D)

def _head_pose(lm, shape):
    h,w=shape[:2]
    m=np.array([[0,0,0],[0,-330,-65],[-225,170,-135],[225,170,-135],
                [-150,-150,-125],[150,-150,-125]],dtype=np.float64)
    p=np.array([[lm[i].x*w,lm[i].y*h] for i in [1,152,33,263,61,291]],dtype=np.float64)
    fl=w; cam=np.array([[fl,0,w/2],[0,fl,h/2],[0,0,1]],dtype=np.float64)
    ok,rv,_=cv2.solvePnP(m,p,cam,np.zeros((4,1)))
    if not ok: return None,None,None
    rm,_=cv2.Rodrigues(rv); a,*_=cv2.RQDecomp3x3(rm)
    return a[0]*360, a[1]*360, a[2]*360


# MediaPipe 전역 초기화
_mp_face = mp.solutions.face_mesh
_face_mesh = _mp_face.FaceMesh(
    max_num_faces=1, refine_landmarks=True,
    min_detection_confidence=0.5, min_tracking_confidence=0.5
)


def face_analysis_agent(state: DMSState) -> DMSState:
    """
    Node 1: MediaPipe로 얼굴 랜드마크 분석
    EAR / MAR / Head Pose 추출
    """
    if state['frame'] is None:
        return {**state, 'face_detected': False}

    rgb = cv2.cvtColor(state['frame'], cv2.COLOR_BGR2RGB)
    res = _face_mesh.process(rgb)

    if not res.multi_face_landmarks:
        return {**state, 'face_detected': False,
                'ear': None, 'mar': None, 'pitch': None, 'yaw': None}

    lm = res.multi_face_landmarks[0].landmark
    h, w = state['frame'].shape[:2]

    le = [(lm[i].x*w, lm[i].y*h) for i in LEFT_EYE]
    re = [(lm[i].x*w, lm[i].y*h) for i in RIGHT_EYE]
    mo = [(lm[i].x*w, lm[i].y*h) for i in MOUTH]

    ear   = (_ear(le) + _ear(re)) / 2.0
    mar   = _mar(mo)
    pitch, yaw, _ = _head_pose(lm, state['frame'].shape)

    # PERCLOS (30초 윈도우)
    history = list(state.get('ear_history', []))
    history.append(ear)
    if len(history) > 30*30: history = history[-30*30:]
    perclos = sum(1 for e in history if e < EAR_THRESH) / len(history)

    return {
        **state,
        'face_detected': True,
        'ear':    round(ear, 4),
        'mar':    round(mar, 4),
        'pitch':  round(pitch, 2) if pitch else None,
        'yaw':    round(yaw, 2)   if yaw   else None,
        'perclos': round(perclos, 4),
        'ear_history': history
    }


print('Node 1: face_analysis_agent 정의 완료')

### 노드 2: Object Detection Agent

In [ ]:
from ultralytics import YOLO

_yolo = YOLO('yolov8n.pt')
DANGEROUS = {67: 'cell phone', 73: 'book'}


def object_detection_agent(state: DMSState) -> DMSState:
    """
    Node 2: YOLOv8로 위험 객체 감지 (휴대폰 등)
    """
    if state['frame'] is None:
        return {**state, 'detected_objects': []}

    results = _yolo(state['frame'], verbose=False)[0]
    objects = []
    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        if cls_id in DANGEROUS and conf >= 0.45:
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            objects.append({
                'class':      DANGEROUS[cls_id],
                'confidence': round(conf, 3),
                'bbox':       [x1,y1,x2,y2]
            })

    return {**state, 'detected_objects': objects}


print('Node 2: object_detection_agent 정의 완료')

### 노드 3: State Classifier Agent

In [ ]:
def state_classifier_agent(state: DMSState) -> DMSState:
    """
    Node 3: 수집된 지표를 분석해 위험 상태 분류
    """
    if not state['face_detected']:
        return {**state,
                'is_drowsy':False,'is_yawning':False,
                'is_distracted':False,'has_danger_obj':False,
                'risk_count':1}

    is_drowsy    = (state['ear'] or 1.0) < EAR_THRESH or (state['perclos'] or 0) > 0.15
    is_yawning   = (state['mar'] or 0)  > MAR_THRESH
    is_distracted = (abs(state['yaw'] or 0) > 30) or ((state['pitch'] or 0) > 20)
    has_danger   = len(state['detected_objects']) > 0

    risk_count = sum([is_drowsy, is_yawning, is_distracted, has_danger])

    return {
        **state,
        'is_drowsy':     is_drowsy,
        'is_yawning':    is_yawning,
        'is_distracted': is_distracted,
        'has_danger_obj': has_danger,
        'risk_count':    risk_count
    }


print('Node 3: state_classifier_agent 정의 완료')

### 노드 4: Alert Manager Agent

In [ ]:
def alert_manager_agent(state: DMSState) -> DMSState:
    """
    Node 4: 경고 레벨 결정 (0~3)
    """
    risk  = state['risk_count']
    obj   = state['detected_objects']
    perc  = state['perclos'] or 0

    if state['has_danger_obj']:
        level  = 3
        reason = f"위험 물체 감지: {obj[0]['class']}"
    elif perc > 0.15 or risk >= 3:
        level  = 3
        reason = '심각한 졸음 / 복합 위험'
    elif risk == 2:
        level  = 2
        reason = '복합 위험 신호 감지'
    elif risk == 1:
        level  = 1
        reason = '주의 필요'
    else:
        level  = 0
        reason = '정상 운전 중'

    return {**state, 'alert_level': level, 'alert_reason': reason}


print('Node 4: alert_manager_agent 정의 완료')

### 노드 5: LLM Reasoning Agent (Ollama)

Ollama가 설치된 경우 실제 LLM, 없으면 Rule-based fallback 사용.

In [ ]:
def _check_ollama() -> bool:
    """Ollama 서비스 실행 중인지 확인"""
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:11434', timeout=2)
        return True
    except:
        return False


def _rule_based_message(state: DMSState) -> str:
    """Ollama 없을 때 Rule-based 메시지 생성"""
    level = state['alert_level']
    msgs = {
        0: '운전 상태 양호합니다. 안전 운전하세요.',
        1: '주의가 필요합니다. 집중력을 유지해주세요.',
        2: '위험 신호가 감지되었습니다! 잠시 휴식을 취하세요.',
        3: '즉시 차량을 안전한 곳에 정차하세요! 매우 위험합니다!'
    }
    reasons = []
    if state['is_drowsy']:    reasons.append('눈 감김/졸음')
    if state['is_yawning']:   reasons.append('하품')
    if state['is_distracted']:reasons.append('전방 주시 이탈')
    if state['has_danger_obj']:reasons.append('위험 물체')
    base = msgs[level]
    if reasons:
        base += f" (감지: {', '.join(reasons)})"
    return base


def llm_reasoning_agent(state: DMSState) -> DMSState:
    """
    Node 5: LLM으로 상황 판단 메시지 생성
    Ollama 없으면 Rule-based fallback
    """
    # 레벨 0이면 LLM 호출 안 함 (비용/속도 최적화)
    if state['alert_level'] == 0:
        return {**state, 'llm_message': '정상 운전 중입니다.'}

    if not _check_ollama():
        # Fallback
        msg = _rule_based_message(state)
        return {**state, 'llm_message': f'[Rule] {msg}'}

    # Ollama LLM 호출
    try:
        llm = ChatOllama(model='qwen2.5:7b', temperature=0.3)
        prompt = f"""당신은 차량 운전자 모니터링 시스템입니다.
현재 감지된 운전자 상태:
- 경고 레벨: {state['alert_level']} (0=정상, 3=위험)
- 원인: {state['alert_reason']}
- EAR(눈감김): {state['ear']}
- 졸음: {state['is_drowsy']}, 하품: {state['is_yawning']}, 부주의: {state['is_distracted']}
- PERCLOS: {state['perclos']}

한 문장으로 운전자에게 경고 메시지를 작성하세요. 간결하고 명확하게."""

        response = llm.invoke([HumanMessage(content=prompt)])
        return {**state, 'llm_message': f'[LLM] {response.content}'}
    except Exception as e:
        msg = _rule_based_message(state)
        return {**state, 'llm_message': f'[Rule] {msg}'}


print('Node 5: llm_reasoning_agent 정의 완료')
print(f'Ollama 상태: {"실행 중" if _check_ollama() else "미설치/미실행 (Rule-based 사용)"}')


## 3. LangGraph 파이프라인 조립

노드들을 연결해서 실제 그래프를 만듭니다.

In [ ]:
def build_dms_graph():
    """DMS LangGraph 파이프라인 빌드"""
    graph = StateGraph(DMSState)

    # 노드 등록
    graph.add_node('face_analysis',     face_analysis_agent)
    graph.add_node('object_detection',  object_detection_agent)
    graph.add_node('state_classifier',  state_classifier_agent)
    graph.add_node('alert_manager',     alert_manager_agent)
    graph.add_node('llm_reasoning',     llm_reasoning_agent)

    # 엣지 연결
    graph.set_entry_point('face_analysis')
    graph.add_edge('face_analysis',    'object_detection')
    graph.add_edge('object_detection', 'state_classifier')
    graph.add_edge('state_classifier', 'alert_manager')
    graph.add_edge('alert_manager',    'llm_reasoning')
    graph.add_edge('llm_reasoning',    END)

    return graph.compile()


dms_app = build_dms_graph()
print('DMS LangGraph 파이프라인 빌드 완료')
print('\n파이프라인 순서:')
print('  face_analysis → object_detection → state_classifier → alert_manager → llm_reasoning → END')

## 4. 테스트 — 더미 프레임으로 파이프라인 검증

In [ ]:
# 웹캠 없어도 테스트 가능한 더미 프레임 생성
def make_dummy_frame(w=640, h=480):
    """간단한 테스트용 더미 프레임"""
    frame = np.zeros((h, w, 3), dtype=np.uint8)
    frame[:] = (100, 100, 100)
    return frame


# 파이프라인 실행 테스트
test_frame = make_dummy_frame()
start_state = initial_state(frame=test_frame, frame_id=1)

print('파이프라인 실행 중...')
result = dms_app.invoke(start_state)

print('\n=== 파이프라인 실행 결과 ===')
print(f'얼굴 감지: {result["face_detected"]}')
print(f'EAR: {result["ear"]}')
print(f'객체 감지: {result["detected_objects"]}')
print(f'경고 레벨: {result["alert_level"]} — {result["alert_reason"]}')
print(f'LLM 메시지: {result["llm_message"]}')
print('\n파이프라인 정상 작동 확인!')

## 5. Ollama 설치 안내 (미설치 시)

4080 Super에서 무료로 돌릴 수 있습니다.

In [ ]:
if not _check_ollama():
    print('='*50)
    print('Ollama 미설치 상태 — Rule-based fallback 사용 중')
    print('='*50)
    print()
    print('[Ollama 설치 방법]')
    print('1. https://ollama.com 접속 → Download for Windows')
    print('2. 설치 후 터미널에서:')
    print('   ollama pull qwen2.5:7b     # 텍스트 모델 (4.7GB)')
    print('   ollama pull llava:13b      # 비전 모델 (8GB, 선택)')
    print()
    print('설치 후 이 셀을 다시 실행하면 LLM 자동 연동됩니다.')
else:
    print('Ollama 실행 중 — LLM 연동 완료!')

## 정리

| 노드 | 역할 | 입력 → 출력 |
|------|------|-------------|
| face_analysis | MediaPipe 분석 | frame → EAR/MAR/Yaw |
| object_detection | YOLOv8 감지 | frame → objects |
| state_classifier | 위험 분류 | 지표들 → bool flags |
| alert_manager | 레벨 결정 | flags → level 0~3 |
| llm_reasoning | 메시지 생성 | level → 경고 문장 |

**다음 Notebook 04**: 웹캠 실시간 연동 + 최종 통합